# Free-Threaded Python: Progress Across Parallel Threads

Free-threaded builds of Python (3.13+, such as `python3.14t`) can run threads in parallel without the GIL. psnek works the same on both kinds of build, and this notebook shows what changes without the GIL:

1. CPU-bound `pmap` on plain threads gets faster with more workers.
2. Bars started by worker threads are drawn as a tree.
3. `update()` from many threads at once stays exact and cheap.

This notebook needs a kernel running a free-threaded build, for example:

```bash
uv venv --python 3.14t ~/.venvs/psnek-ft
VIRTUAL_ENV=~/.venvs/psnek-ft uv pip install -e . ipykernel
~/.venvs/psnek-ft/bin/python -m ipykernel install --user --name psnek-ft \
    --display-name "psnek (3.14t)"
```

On a regular build, every cell still runs, but the threads take turns.

In [1]:
import logging
import os
import sys
import sysconfig
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

import psnek

logger = logging.getLogger("demo")
free_threaded = bool(sysconfig.get_config_var("Py_GIL_DISABLED"))
gil_enabled = sys._is_gil_enabled() if hasattr(sys, "_is_gil_enabled") else True
print(
    f"Python {sys.version.split()[0]}, free-threaded build: {free_threaded}, "
    f"GIL enabled: {gil_enabled}, {os.cpu_count()} CPUs"
)
if gil_enabled:
    logger.warning("the GIL is enabled, so the threads below will take turns")

Python 3.14.0, free-threaded build: True, GIL enabled: False, 12 CPUs


## 1. CPU-Bound Work on Threads

`pmap` uses a thread pool by default. Without the GIL, pure-Python work spreads across cores without needing a `ProcessPoolExecutor`.

In [2]:
def checksum(block: int) -> int:
    """Pretend to checksum ``block`` with pure-Python arithmetic."""
    total = 0
    for value in range(150_000):
        total = (total + value * block) % 1_000_003
    return total


timings = {}
for workers in (1, 2, 4, 8):
    start = time.perf_counter()
    psnek.pmap(checksum, range(200), f"checksum, workers={workers}", workers=workers)
    timings[workers] = time.perf_counter() - start
for workers, seconds in timings.items():
    speedup = timings[1] / seconds
    print(f"workers={workers}: {seconds:5.2f}s ({speedup:.1f}x one worker)")

# checksum, workers=1 ###################  100.0%    200/200         0:00    246it/s  done
# checksum, workers=2 ###################  100.0%    200/200         0:00    479it/s  done
# checksum, workers=4 ###################  100.0%    200/200         0:00    800it/s  done
# checksum, workers=8 ###################  100.0%    200/200         0:00   1147it/s  done

workers=1:  0.81s (1.0x one worker)
workers=2:  0.42s (2.0x one worker)
workers=4:  0.25s (3.3x one worker)
workers=8:  0.17s (4.7x one worker)


## 2. A Bar per Worker Thread

A bar started by a worker thread is drawn below the top-level bar, beside the bars of the other workers. Here, four threads each verify a shard of 40 blocks.

In [3]:
def verify_shard(shard: int) -> int:
    """Checksum the 40 blocks of ``shard``, with a bar of its own."""
    blocks = range(shard * 40, (shard + 1) * 40)
    return sum(checksum(block) for block in psnek.track(blocks, f"shard {shard}"))


with (
    psnek.Bar(
        total=8, desc="verify shards", options=psnek.Options(unit="shard")
    ) as bar,
    ThreadPoolExecutor(4) as pool,
):
    futures = [pool.submit(verify_shard, shard) for shard in range(8)]
    for future in as_completed(futures):
        future.result()
        bar.update()

# verify shards #######################  100.0%        8/8         0:00  23.8shard/s  done

## 3. `update()` from Many Threads

Without the GIL, each thread that updates a bar gets its own queue, so the threads don't contend for one lock. Here, 8 threads call `update()` after every item, and the count stays exact.

In [4]:
ITEMS_PER_THREAD = 25_000
THREADS = 8
TOTAL = ITEMS_PER_THREAD * THREADS


def process(items: int, bar: psnek.Bar | None) -> None:
    """Process ``items`` small items, reporting each to ``bar`` if given."""
    for _ in range(items):
        sum(range(600))
        if bar is not None:
            bar.update()


def run_threads(bar: psnek.Bar | None) -> float:
    """Run ``process`` on every thread at once and return the seconds taken."""
    threads = [
        threading.Thread(target=process, args=(ITEMS_PER_THREAD, bar))
        for _ in range(THREADS)
    ]
    start = time.perf_counter()
    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join()
    return time.perf_counter() - start


bare = min(run_threads(None) for _ in range(3))
with psnek.Bar(total=3 * TOTAL, desc="8 threads, update() per item") as bar:
    timed = min(run_threads(bar) for _ in range(3))
per_item = bare / ITEMS_PER_THREAD * 1e6
print(f"counted {bar.n:,} of {3 * TOTAL:,} items ({per_item:.1f} us each)")
print(f"{bare:.3f}s without a bar, {timed:.3f}s with update() per item")

# 8 threads, update() per item ##########  100.0%  600000/600000         0:00  1.95Mit/s  done

counted 600,000 of 600,000 items (3.6 us each)
0.090s without a bar, 0.093s with update() per item


For items that take only a few microseconds each, call `bar.update(len(batch))` once per batch instead, as `pmap` does.